# Epic 3 — Find Local Digital Help
## Phase 1: Combine AIHW Aged Care Services + Library Datasets


---

### Purpose
Combine the cleaned AIHW Aged Care Service List and the cleaned Libraries dataset 
into a single unified CSV for use in the Epic 3 Local Help Finder feature.

This is a **union (stack)** — not a join/merge — because the two datasets represent 
different venue types, not the same venues with different attributes.

### Input Files
| File | Records | Description |
|------|---------|-------------|
| `cleaned_AIHW_ServiceList.csv` | 5,376 | Aged care services — national coverage |
| `cleaned_Libraries_ALL.csv` | 835 | Libraries — VIC, QLD, TAS, WA, SA |

### Output
`src/data/clean_data/cleaned_LocalHelp_ALL.csv`

### Target Schema
| Column | Description |
|--------|-------------|
| `name` | Venue name |
| `address` | Street address |
| `suburb` | Suburb or town |
| `state` | State abbreviation |
| `postcode` | 4-digit postcode as string |
| `latitude` | Decimal latitude |
| `longitude` | Decimal longitude |
| `phone` | Contact phone (null where unavailable) |
| `opening_hours` | Opening hours (null where unavailable) |
| `venue_type` | Venue category for UI filter chips |
| `source` | Source dataset for data lineage |

## 1. Import Libraries

In [58]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 2. Set File Paths

In [59]:
# Navigate from src/notebooks up to src/
src_base_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
clean_path    = os.path.join(src_base_path, 'data', 'clean_data')

AIHW_PATH    = os.path.join(clean_path, 'cleaned_AIHW_ServiceList.csv')
LIB_PATH     = os.path.join(clean_path, 'cleaned_Libraries_ALL.csv')
OUTPUT_PATH  = os.path.join(clean_path, 'Combined_LocalHelp_ALL.csv')

print('Clean data path :', clean_path)
print('AIHW file exists:', os.path.exists(AIHW_PATH))
print('Lib file exists :', os.path.exists(LIB_PATH))

Clean data path : c:\Monash\SEM4\FIT-5120-main\FIT-5120-Main-project\src\data\clean_data
AIHW file exists: True
Lib file exists : True


## 3. Load Both Cleaned Datasets

In [60]:
df_aihw = pd.read_csv(AIHW_PATH, dtype={'Physical Post Code': str})
df_lib  = pd.read_csv(LIB_PATH,  dtype={'postcode': str})

print(f'AIHW records  : {df_aihw.shape[0]} rows x {df_aihw.shape[1]} columns')
print(f'Library records: {df_lib.shape[0]} rows x {df_lib.shape[1]} columns')
print(f'\nAIHW columns  : {list(df_aihw.columns)}')
print(f'\nLib columns   : {list(df_lib.columns)}')

AIHW records  : 5376 rows x 24 columns
Library records: 835 rows x 11 columns

AIHW columns  : ['Service Name', 'Physical Address', 'Physical Suburb', 'Physical State', 'Physical Post Code', '2018 Aged Care Planning Region (ACPR)', 'Care Type', 'Residential Places', 'Home Care Places', 'Restorative Care Places', 'Provider Name', 'Organisation Type', 'ABS Remoteness', '2019 MMM Code', '2016 SA2 Code', '2016 SA2 Name', '2016 SA3 Code', '2016 SA3 Name', '2023 LGA Name', '2023 LGA Code', '2017 PHN Code', '2017 PHN Name', 'Latitude', 'Longitude']

Lib columns   : ['name', 'address', 'suburb', 'state', 'postcode', 'latitude', 'longitude', 'phone', 'opening_hours', 'venue_type', 'source']


## 4. Pre-Combination EDA

Confirm the key fields in each dataset before standardising schemas.

### 4.1 AIHW  Care Type Distribution

In [61]:
print('Care Type breakdown:')
print(df_aihw['Care Type'].value_counts().to_string())
print('\nState breakdown:')
print(df_aihw['Physical State'].value_counts().to_string())
print('\nNull counts (key fields):')
key_cols = ['Service Name','Physical Address','Physical Suburb',
            'Physical State','Physical Post Code','Latitude','Longitude']
print(df_aihw[key_cols].isnull().sum().to_string())

Care Type breakdown:
Care Type
Residential                                                         2590
Home Care                                                           2363
Multi-Purpose Service                                                183
Short-Term Restorative Care (STRC)                                   126
Transition Care                                                       67
National Aboriginal and Torres Strait Islander Aged Care Program      47

State breakdown:
Physical State
NSW    1703
VIC    1415
QLD    1013
WA      541
SA      413
TAS     151
NT       71
ACT      69

Null counts (key fields):
Service Name          0
Physical Address      0
Physical Suburb       0
Physical State        0
Physical Post Code    0
Latitude              0
Longitude             0


### 4.2 Libraries State and Venue Type Distribution

In [62]:
print('State breakdown:')
print(df_lib['state'].value_counts().to_string())
print('\nVenue type breakdown:')
print(df_lib['venue_type'].value_counts().to_string())
print('\nNull counts (key fields):')
print(df_lib[['name','address','suburb','state','postcode',
              'latitude','longitude','phone','opening_hours']].isnull().sum().to_string())

State breakdown:
state
VIC    281
WA     233
SA     156
QLD    119
TAS     46

Venue type breakdown:
venue_type
Library    835

Null counts (key fields):
name               0
address            0
suburb             0
state              0
postcode           1
latitude           0
longitude          0
phone              7
opening_hours    699


## 5. Standardise AIHW to Target Schema

The AIHW dataset has 24 columns with different names to the library schema.
We select only the columns needed for Epic 3 and rename them to match.

We also map `Care Type` to readable `venue_type` labels for the UI filter chips.
AIHW has no phone or opening hours data — we fill these with standardised
fallback strings so the frontend can display them consistently.

In [63]:
# Map AIHW Care Type values to readable venue_type labels
# These will appear as filter chip labels in the UI
care_type_map = {
    'Residential'                                                        : 'Residential Aged Care',
    'Home Care'                                                          : 'Home Care Service',
    'Multi-Purpose Service'                                              : 'Multi-Purpose Service',
    'Short-Term Restorative Care (STRC)'                                 : 'Restorative Care',
    'Transition Care'                                                    : 'Transition Care',
    'National Aboriginal and Torres Strait Islander Aged Care Program'   : 'Indigenous Aged Care'
}

df_aihw['venue_type'] = df_aihw['Care Type'].map(care_type_map)

print('venue_type distribution after mapping:')
print(df_aihw['venue_type'].value_counts().to_string())
print(f'\nUnmapped (null) venue_type: {df_aihw["venue_type"].isnull().sum()}')

venue_type distribution after mapping:
venue_type
Residential Aged Care    2590
Home Care Service        2363
Multi-Purpose Service     183
Restorative Care          126
Transition Care            67
Indigenous Aged Care       47

Unmapped (null) venue_type: 0


In [64]:
# Select and rename AIHW columns to match target schema
aihw_std = pd.DataFrame({
    'name'         : df_aihw['Service Name'],
    'address'      : df_aihw['Physical Address'],
    'suburb'       : df_aihw['Physical Suburb'],
    'state'        : df_aihw['Physical State'],
    'postcode'     : df_aihw['Physical Post Code'].astype(str).str.strip(),
    'latitude'     : df_aihw['Latitude'],
    'longitude'    : df_aihw['Longitude'],
    # AIHW has no phone data — use consistent fallback string
    'phone'        : 'Phone number not available',
    # AIHW has no opening hours — use consistent fallback string
    'opening_hours': 'Opening hours not available',
    'venue_type'   : df_aihw['venue_type'],
    'source'       : 'AIHW_ServiceList'
})

print(f'AIHW standardised shape: {aihw_std.shape}')
print(f'\nNull counts after standardising:')
print(aihw_std.isnull().sum().to_string())
print(f'\nSample row:')
print(aihw_std.iloc[0].to_string())

AIHW standardised shape: (5376, 11)

Null counts after standardising:
name             0
address          0
suburb           0
state            0
postcode         0
latitude         0
longitude        0
phone            0
opening_hours    0
venue_type       0
source           0

Sample row:
name             Braidwood Multi-Purpose Service
address                      77 Monkittee Street
suburb                                 BRAIDWOOD
state                                        NSW
postcode                                    2622
latitude                              -35.442764
longitude                             149.805407
phone                 Phone number not available
opening_hours        Opening hours not available
venue_type                 Multi-Purpose Service
source                          AIHW_ServiceList


## 6. Standardise Libraries to Target Schema

The library dataset already matches the target schema.
We only need to fill null phone and opening_hours with the same
standardised fallback strings used in the AIHW dataset so the
frontend handles them consistently.

In [65]:
lib_std = df_lib.copy()

# Fill null phones with consistent fallback
lib_std['phone'] = lib_std['phone'].fillna('Contact not available')

# Fill null opening hours with consistent fallback
lib_std['opening_hours'] = lib_std['opening_hours'].fillna('Call ahead to confirm hours')

# Confirm schema matches AIHW standardised columns
SCHEMA = ['name','address','suburb','state','postcode',
          'latitude','longitude','phone','opening_hours',
          'venue_type','source']

print(f'Library standardised shape: {lib_std.shape}')
print(f'\nNull counts after standardising:')
print(lib_std[SCHEMA].isnull().sum().to_string())
print(f'\nSample row:')
print(lib_std[SCHEMA].iloc[0].to_string())

Library standardised shape: (835, 11)

Null counts after standardising:
name             0
address          0
suburb           0
state            0
postcode         1
latitude         0
longitude        0
phone            0
opening_hours    0
venue_type       0
source           0

Sample row:
name                     Albert Park Library
address                  319 Montague Street
suburb                           Albert Park
state                                    VIC
postcode                                3206
latitude                              -37.84
longitude                             144.96
phone                              9209 6622
opening_hours    Call ahead to confirm hours
venue_type                           Library
source                         Libraries_VIC


## 7. Combine Both Datasets

Stack the two standardised datasets using `pd.concat`.
Both datasets are aligned to the same 11-column schema before stacking.

In [66]:
SCHEMA = ['name','address','suburb','state','postcode',
          'latitude','longitude','phone','opening_hours',
          'venue_type','source']

combined = pd.concat([
    aihw_std[SCHEMA],
    lib_std[SCHEMA]
], ignore_index=True)

print(f'Combined shape : {combined.shape[0]} rows x {combined.shape[1]} columns')
print(f'\nVenue type breakdown:')
print(combined['venue_type'].value_counts().to_string())
print(f'\nState breakdown:')
print(combined['state'].value_counts().to_string())

Combined shape : 6211 rows x 11 columns

Venue type breakdown:
venue_type
Residential Aged Care    2590
Home Care Service        2363
Library                   835
Multi-Purpose Service     183
Restorative Care          126
Transition Care            67
Indigenous Aged Care       47

State breakdown:
state
NSW    1703
VIC    1696
QLD    1132
WA      774
SA      569
TAS     197
NT       71
ACT      69


## 8. Post-Combination Validation

In [67]:
# Check nulls across combined dataset
null_pct = (combined.isnull().sum() / len(combined) * 100).round(1)
print('Null Value Summary:')
print(pd.DataFrame({
    'Null Count': combined.isnull().sum(),
    'Null %'    : null_pct
}).to_string())

Null Value Summary:
               Null Count  Null %
name                    0     0.0
address                 0     0.0
suburb                  0     0.0
state                   0     0.0
postcode                1     0.0
latitude                0     0.0
longitude               0     0.0
phone                   0     0.0
opening_hours           0     0.0
venue_type              0     0.0
source                  0     0.0


In [68]:
# Confirm all coordinates are within Australia's geographic bounds
out_of_bounds = combined[
    (combined['latitude']  < -44) | (combined['latitude']  > -10) |
    (combined['longitude'] < 113) | (combined['longitude'] > 154)
]
print(f'Records outside Australia bounds : {len(out_of_bounds)}')
print(f'Latitude  range : {combined["latitude"].min():.4f} to {combined["latitude"].max():.4f}')
print(f'Longitude range : {combined["longitude"].min():.4f} to {combined["longitude"].max():.4f}')

Records outside Australia bounds : 5
Latitude  range : -43.3177 to -10.4216
Longitude range : 96.8293 to 167.9548


In [69]:
# Confirm source breakdown — useful for data lineage
print('Source breakdown:')
print(combined['source'].value_counts().to_string())

Source breakdown:
source
AIHW_ServiceList    5376
Libraries_VIC        281
Libraries_WA         233
Libraries_SA         156
Libraries_QLD        119
Libraries_TAS         46


In [70]:
# Preview final combined dataset — sample from each source
print('=== Sample AIHW record ===')
print(combined[combined['source'] == 'AIHW_ServiceList'].iloc[0].to_string())
print('\n=== Sample Library record ===')
print(combined[combined['source'] == 'Libraries_VIC'].iloc[0].to_string())

=== Sample AIHW record ===
name             Braidwood Multi-Purpose Service
address                      77 Monkittee Street
suburb                                 BRAIDWOOD
state                                        NSW
postcode                                    2622
latitude                              -35.442764
longitude                             149.805407
phone                 Phone number not available
opening_hours        Opening hours not available
venue_type                 Multi-Purpose Service
source                          AIHW_ServiceList

=== Sample Library record ===
name                     Albert Park Library
address                  319 Montague Street
suburb                           Albert Park
state                                    VIC
postcode                                3206
latitude                              -37.84
longitude                             144.96
phone                              9209 6622
opening_hours    Call ahead to confirm hour

In [71]:
combined.head(10)

,name,address,suburb,state,postcode,latitude,longitude,phone,opening_hours,venue_type,source
0,Braidwood Multi-Purpose Service,77 Monkittee Street,BRAIDWOOD,NSW,2622,-35.442764,149.805407,Phone number not available,Opening hours not available,Multi-Purpose Service,AIHW_ServiceList
1,Dellacourt,42 NICHOLSON Place,WEST ALBURY,NSW,2640,-36.075401,146.890896,Phone number not available,Opening hours not available,Residential Aged Care,AIHW_ServiceList
2,Brerrina Multi-Purpose Service,56 DOYLE Street,BRERRINA,NSW,2839,-29.961591,146.864313,Phone number not available,Opening hours not available,Multi-Purpose Service,AIHW_ServiceList
3,BaptistCare Maranoa Centre - Alstonville,15 The Avenue -,ALSTONVILLE,NSW,2477,-28.840536,153.436634,Phone number not available,Opening hours not available,Residential Aged Care,AIHW_ServiceList
4,Urana Multi-Purpose Service,127-129 Princess Street,URANA,NSW,2645,-35.325391,146.267697,Phone number not available,Opening hours not available,Multi-Purpose Service,AIHW_ServiceList
5,Baradine Multi-Purpose Service,5-9 Macquarie Street,BARADINE,NSW,2396,-30.945519,149.067008,Phone number not available,Opening hours not available,Multi-Purpose Service,AIHW_ServiceList
6,Uniting Autumn Lodge Butler Street,50 Butler Street,ARMIDALE,NSW,2350,-30.503834,151.658370,Phone number not available,Opening hours not available,Residential Aged Care,AIHW_ServiceList
7,Macquarie Lodge Aged Care Plus Centre,171 Wollongong Road,ARNCLIFFE,NSW,2205,-33.939656,151.134856,Phone number not available,Opening hours not available,Residential Aged Care,AIHW_ServiceList
8,Urbenville Multi-Purpose Service,45 Beaury Street,URBENVILLE,NSW,2475,-28.471555,152.543270,Phone number not available,Opening hours not available,Multi-Purpose Service,AIHW_ServiceList
9,Ashfield Terrace Care Community,8-10 Clissold Street,ASHFIELD,NSW,2131,-33.895950,151.127894,Phone number not available,Opening hours not available,Residential Aged Care,AIHW_ServiceList


## 9. Export Combined Dataset

Save to `cleaned_LocalHelp_ALL.csv` — this is the primary data file
for the Epic 3 Find Local Digital Help feature.

In [72]:
combined = combined.reset_index(drop=True)
combined.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print(f'Saved to      : {OUTPUT_PATH}')
print(f'Final shape   : {combined.shape[0]} rows x {combined.shape[1]} columns')
print(f'\nFinal venue_type breakdown:')
print(combined['venue_type'].value_counts().to_string())
print(f'\nFinal state breakdown:')
print(combined['state'].value_counts().to_string())

Saved to      : c:\Monash\SEM4\FIT-5120-main\FIT-5120-Main-project\src\data\clean_data\Combined_LocalHelp_ALL.csv
Final shape   : 6211 rows x 11 columns

Final venue_type breakdown:
venue_type
Residential Aged Care    2590
Home Care Service        2363
Library                   835
Multi-Purpose Service     183
Restorative Care          126
Transition Care            67
Indigenous Aged Care       47

Final state breakdown:
state
NSW    1703
VIC    1696
QLD    1132
WA      774
SA      569
TAS     197
NT       71
ACT      69


## 10. Coordinate Validation via Postcode Cross-Check

For each record with a valid postcode, we look up the reference lat/lon
for that postcode using pgeocode and compute the distance to the stored
coordinates. Records more than 50km from their postcode centroid are
flagged as potentially incorrect and logged for review.

This does not mean flagged records are wrong  a postcode centroid can be
far from the actual venue in large rural postcodes. But it surfaces
any genuine data errors worth investigating.

In [73]:
import pgeocode
import numpy as np

# Load Australian postcode reference
nomi = pgeocode.Nominatim('AU')
postcode_ref = nomi._data[['postal_code', 'latitude', 'longitude']].dropna()
postcode_ref = postcode_ref.rename(columns={
    'postal_code': 'postcode',
    'latitude'   : 'ref_lat',
    'longitude'  : 'ref_lon'
})
postcode_ref['postcode'] = postcode_ref['postcode'].astype(str).str.strip()

# Load combined dataset
combined = pd.read_csv(OUTPUT_PATH, dtype={'postcode': str})
combined['postcode'] = combined['postcode'].astype(str).str.strip()

print(f'Combined records      : {len(combined)}')
print(f'Records with postcode : {combined["postcode"].notna().sum()}')
print(f'Records without       : {combined["postcode"].isna().sum()}')

Combined records      : 6211
Records with postcode : 6211
Records without       : 0


In [74]:
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Calculate distance in km between two lat/lon points
    using the Haversine formula — accounts for Earth's curvature.
    More accurate than Euclidean for distance validation.
    """
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Merge combined dataset with postcode reference lat/lon
validation = combined.merge(postcode_ref, on='postcode', how='left')

# Calculate distance between stored and reference coordinates
validation['distance_km'] = validation.apply(
    lambda row: haversine_km(
        row['latitude'], row['longitude'],
        row['ref_lat'],  row['ref_lon']
    ) if pd.notna(row['ref_lat']) else np.nan,
    axis=1
)

print(f'Records validated     : {validation["distance_km"].notna().sum()}')
print(f'Records unmatched     : {validation["distance_km"].isna().sum()}')
print(f'\nDistance summary (km):')
print(validation['distance_km'].describe().round(2).to_string())

Records validated     : 65802
Records unmatched     : 78

Distance summary (km):
count    65802.00
mean        20.68
std         64.78
min          0.00
25%          2.46
50%          6.66
75%         16.60
max       2527.94


In [75]:
# Flag records more than 50km from their postcode centroid
THRESHOLD_KM = 50

flagged = validation[validation['distance_km'] > THRESHOLD_KM].copy()
flagged = flagged.sort_values('distance_km', ascending=False)

print(f'Records within {THRESHOLD_KM}km  : {(validation["distance_km"] <= THRESHOLD_KM).sum()}')
print(f'Records flagged (>{THRESHOLD_KM}km): {len(flagged)}')

if len(flagged) > 0:
    print(f'\nFlagged records — review for potential coordinate errors:')
    print(flagged[['name','suburb','state','postcode',
                   'latitude','longitude',
                   'ref_lat','ref_lon',
                   'distance_km','venue_type']].to_string())
else:
    print('\nNo records flagged — all coordinates are within acceptable range.')

Records within 50km  : 61430
Records flagged (>50km): 4372

Flagged records — review for potential coordinate errors:
                                                                                 name                                      suburb state postcode   latitude   longitude  ref_lat   ref_lon  distance_km             venue_type
63201                                                Cocos West Island Public Library                     Cocos (Keeling) Islands    WA     6799 -12.188390   96.829280 -20.7905  118.8132  2527.941904                Library
63200                                                Cocos West Island Public Library                     Cocos (Keeling) Islands    WA     6799 -12.188390   96.829280 -20.7905  118.8132  2527.941904                Library
63199                                                Cocos Home Island Public Library                     Cocos (Keeling) Islands    WA     6799 -12.118550   96.897860 -20.7905  118.8132  2524.503162              

In [ ]:
df = pd.read_csv(OUTPUT_PATH, dtype={'postcode': str})

print(f"Total records           : {len(df)}")
print(f"Columns                 : {list(df.columns)}")
print()

# ── 1. EXACT DUPLICATES (all 11 columns identical) ──────────────────────────
exact_dupes = df[df.duplicated(keep=False)]
print(f"── 1. Exact duplicates (all columns) ──")
print(f"Duplicate records       : {len(exact_dupes)}")
print(f"Unique duplicate groups : {df.duplicated().sum()}")
if len(exact_dupes) > 0:
    print(exact_dupes[['name','address','suburb','state','venue_type','source']].head(20).to_string())
print()

# ── 2. LOGICAL DUPLICATES (same name + address, case-insensitive) ────────────
df['_name_clean']    = df['name'].str.strip().str.lower()
df['_address_clean'] = df['address'].str.strip().str.lower()

logical_dupes = df[df.duplicated(subset=['_name_clean','_address_clean'], keep=False)]
print(f"── 2. Logical duplicates (same name + address) ──")
print(f"Duplicate records       : {len(logical_dupes)}")
if len(logical_dupes) > 0:
    print(logical_dupes[['name','address','suburb','state','venue_type','source']]
          .sort_values(['_name_clean','_address_clean'])
          .head(30).to_string())
print()

# ── 3. COORDINATE DUPLICATES (same lat/lon, different name) ─────────────────
coord_dupes = df[df.duplicated(subset=['latitude','longitude'], keep=False)]
coord_dupes_diff_name = coord_dupes[
    coord_dupes.duplicated(subset=['latitude','longitude','_name_clean'], keep=False) == False
]
print(f"── 3. Coordinate duplicates (same lat/lon, different name) ──")
print(f"Records sharing coordinates : {len(coord_dupes)}")
print(f"Of those, different names   : {len(coord_dupes_diff_name)}")
if len(coord_dupes_diff_name) > 0:
    print(coord_dupes_diff_name[['name','address','suburb','state','latitude','longitude','venue_type','source']]
          .sort_values(['latitude','longitude'])
          .head(30).to_string())
print()

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("── Summary ──")
print(f"Exact duplicates        : {df.duplicated().sum()}")
print(f"Logical duplicates      : {df.duplicated(subset=['_name_clean','_address_clean']).sum()}")
print(f"Coordinate duplicates   : {df.duplicated(subset=['latitude','longitude']).sum()}")

# cleanup temp columns
df.drop(columns=['_name_clean','_address_clean'], inplace=True)